# Demo Run

Deploy 3 models (seed-2.0-mini, kimi-k3, gemma-4-31b) on 8 videos (4 countries × math+science), first 15 min each, 4 prompt types. Then summarize each video+model combo across 4 perspectives in 60s rolling windows.

## 上下文 -

### editing notes

当修改涉及逐步构建的函数时，除了改最终函数定义，还要回溯改所有中间探索步骤——单行测试、循环原型、`Video(...)` 构造等——凡是涉及被改字段/参数的 cell 都要同步更新，保持探索轨迹和最终代码一致。

- 提出任何代码修改建议时，一律用 diff block 呈现改动。

### docs

### styling

### directory

In [ ]:
!tree

.
├── CONTROLLER.ipynb
├── CONTROLLER_dup1.ipynb
├── CONTROLLER_dup2.ipynb
├── CONTROLLER_dup3.ipynb
├── LICENSE
├── MANIFEST.in
├── README.md
├── _proc
│   ├── 00_core.ipynb
│   ├── _docs
│   │   ├── index.html
│   │   ├── robots.txt
│   │   └── sitemap.xml
│   ├── _quarto.yml
│   ├── index.ipynb
│   ├── nbdev.yml
│   └── styles.css
├── cachy.jsonl
├── db.db
├── db.db-shm
├── db.db-wal
├── db1.db
├── db1.db-shm
├── db1.db-wal
├── db2.db-shm
├── db2.db-wal
├── db3.db
├── db3.db-shm
├── db3.db-wal
├── db_demo.db
├── db_demo.db-shm
├── db_demo.db-wal
├── db_demo2.db
├── db_demo2.db-shm
├── db_demo2.db-wal
├── demo_run.ipynb
├── nbs
│   ├── 00_core.ipynb
│   ├── _quarto.yml
│   ├── db.db
│   ├── db.db-shm
│   ├── db.db-wal
│   ├── index.ipynb
│   ├── nbdev.yml
│   ├── styles.css
│   └── test.jpg
├── pyproject.toml
├── science_transcripts.zip
├── vlm_monitor
│   ├── __init__.py
│   ├── __pycache__
│   │   ├── __init__.cpython-312.pyc
│   │   └── core.cpython-312.pyc
│   ├── _modidx.py
│   

In [ ]:
!tree -L 1 ../data/timss

../data/timss
├── M-AU1
├── M-AU2
├── M-AU3
├── M-AU4
├── M-CZ1
├── M-CZ2
├── M-CZ3
├── M-CZ4
├── M-HK1
├── M-HK2
├── M-HK3
├── M-HK4
├── M-JP1
├── M-JP2
├── M-JP3
├── M-JP4
├── M-NL1
├── M-NL2
├── M-NL3
├── M-NL4
├── M-SW1
├── M-SW2
├── M-SW3
├── M-SW4
├── M-US1
├── M-US2
├── M-US3
├── M-US4
├── S-AU1
├── S-AU2
├── S-AU3
├── S-AU4
├── S-AU5
├── S-CZ1
├── S-CZ2
├── S-CZ3
├── S-CZ4
├── S-CZ5
├── S-JP1
├── S-JP2
├── S-JP3
├── S-JP4
├── S-JP5
├── S-NL1
├── S-NL2
├── S-NL3
├── S-NL4
├── S-NL5
├── S-US1
├── S-US3
├── S-US4
└── S-US5

53 directories, 0 files


# controller

> Control the library from here

Hi there, in all interactions in this dialogue, I want you to refreshingly and invigoratingly guide me Polya style. At the same time, I don't want you to jump ahead of where I am, which includes asking follow-up questions. Simply acknowledge this, that's all.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

收到啦！(｀・ω・´)ゞ

我会用 Polya 的方式来引导你，一步一步走，绝不抢跑～

```json {.usage}
{"model": "accounts/fireworks/models/glm-5p2", "prompt_tokens": 50241, "completion_tokens": 178, "total_tokens": 50419, "cached_tokens": 39983, "cost": 0.03111303}
```


In [ ]:
#| hide
from nbdev import nbdev_export; nbdev_export('./nbs/')

from toolslm.xml import nb2xml
core = nb2xml(Path('./nbs/00_core.ipynb')); core

# from toolslm.xml import files2ctx
# core = files2ctx('./vlm_monitor/core.py'); core

'<notebook><raw id="7bdca9ea"><source>---\nskip_exec: true\n---</raw><md id="277a059d"><source># core\n\n> The source of truth</md><md id="3efaa0ce"><source>> This library follows the [fastai style guide](https://docs.fast.ai/dev/style.html), and is crafted with [nbdev](https://nbdev.fast.ai/).</md><md id="654d9533"><source>This is a general purpose library that allows you to use a VLM (Vision Language Model) to thoroughly describe the contents of a video, with subtitles included for additional context.\n\nIn essence, this library provides a second set of eyes.\n\nThe output is a database object containing the video description. \n\nIf you want to directly run this notebook, you want to have an `OPENROUTER_API_KEY` set.</md><md id="0588b935"><source>### Preface</md><md id="5d37f23f"><source>This is a library that allows you to thoroughly describe what occurs in a video.\n\n**Concisely:**\n\n- A VLM describes the video frame by frame. Each time, it is provided with an empty history toge

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

### Demo Run

Deploy 3 models (seed-2.0-mini, kimi-k3, gemma-4-31b) on 8 videos (4 countries × math+science), first 15 min each, 4 prompt types. Then summarize each video+model combo across 4 perspectives in 60s rolling windows.

---

In [ ]:
from vlm_monitor.core import *
from cachy import enable_cachy; enable_cachy()

In [ ]:
# # !rm -rf db_demo2.db
# db = init_db('db_demo3.db')
# dpath = Path('../data/timss/')
# dpaths = filter_paths(dpath.ls()).sorted(lambda o: (o.stem[:-1], o.stem[-1]))
# populate_db(db, dpaths)
# len(db.t.video()), len(db.t.frame())

HTML(
<style>
    progress { appearance: none; border: none; border-radius: 4px; width: 300px;
        height: 20px; vertical-align: middle; background: #e0e0e0; }

    progress::-webkit-progress-bar { background: #e0e0e0; border-radius: 4px; }
    progress::-webkit-progress-value { background: #2196F3; border-radius: 4px; }
    progress::-moz-progress-bar { background: #2196F3; border-radius: 4px; }

    progress:not([value]) {
        background: repeating-linear-gradient(45deg, #7e7e7e, #7e7e7e 10px, #5c5c5c 10px, #5c5c5c 20px); }

    progress.progress-bar-interrupted::-webkit-progress-value { background: #F44336; }
    progress.progress-bar-interrupted::-moz-progress-value { background: #F44336; }
    progress.progress-bar-interrupted::-webkit-progress-bar { background: #F44336; }
    progress.progress-bar-interrupted::-moz-progress-bar { background: #F44336; }
    progress.progress-bar-interrupted { background: #F44336; }    

    table.fastprogress { border-collapse: collapse; margin: 1em 0; font-size: 0.9em; }
    table.fastprogress th, table.fastprogress td { padding: 8px 12px; border: 1px solid #ddd; text-align: left; }
    table.fastprogress thead tr { background: #f8f9fa; font-weight: bold; }
    table.fastprogress tbody tr:nth-of-type(even) { background: #f8f9fa; }
</style>
)

<div></div>

(52, 149880)

In [ ]:
from fastlite import *
db = database('db_demo3.db')
for t in db.t: t.dataclass()

In [ ]:
models = AttrDict(
    seed2p0mini=AttrDict(name='bytedance-seed/seed-2.0-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    # kimi3=AttrDict(name='moonshotai/kimi-k3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    gemma31b=AttrDict(name='google/gemma-4-31b-it', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
)

In [ ]:
caveman_prompt = '''
Respond terse like smart caveman. All technical substance stay. Only fluff die.

## Persistence

ACTIVE EVERY RESPONSE. No revert after many turns. No filler drift. Still active if unsure.

## Rules

Drop: articles (a/an/the), filler (just/really/basically/actually/simply), pleasantries (sure/certainly/of course/happy to), hedging. Fragments OK. Short synonyms (big not extensive, fix not "implement a solution for"). Technical terms exact. Code blocks unchanged. Errors quoted exact.

Pattern: `[thing] [action] [reason]. [next step.]`

Not: "Sure! I'd be happy to help you with that. The issue you're experiencing is likely caused by..."
Yes: "Bug in auth middleware. Token expiry check use `<` not `<=`. Fix:"

## Intensity

Example — "Why React component re-render?"
- "New object ref each render. Inline object prop = new ref = re-render. Wrap in `useMemo`."

Example — "Explain database connection pooling."
- "Pool reuse open DB connections. No new connection per request. Skip handshake overhead."

## Auto-Clarity

Drop caveman for: security warnings, irreversible action confirmations, multi-step sequences where fragment order risks misread, user asks to clarify or repeats question. Resume caveman after clear part done.

Example — destructive op:
> **Warning:** This will permanently delete all rows in the `users` table and cannot be undone.
> ```sql
> DROP TABLE users;
> ```
> Caveman resume. Verify backup exist first.
'''

prompt_template = '''
You are an observer of a classroom. You will be provided with a recording of a class, and your task at hand is to neutrally describe the requested feature of the classroom. You need to ground the descriptions with evidence supported by the supplied frame, subtitle, or other input provided to you.

Here, you are assigned to observe:

{}

Report only what is directly observable, and what was requested, recalling the role you have been assigned.

No speculation, no assumptions, no decorative language, no conclusions, no inferences, no inventions, no external knowledge and no observations beyond what was specifically requested.

If there is ambiguity, state so.
'''

In [ ]:
vlm_prompt = '''
- Detail the environment presented in the given frame.

- Describe observable use of the whiteboard, blackboard, worksheets, projector, textbooks, computers, diagrams, physical objects, or other instructional resources and how they are being used. If these objects contain contents, describe the topic, task, procedure, problem, concept, application, representation, or solution strategy present on the contents.

- Detail the teacher presented in the given frame, concretely describing their actions, when they are visible, audible, or explicitly mentioned. Such descriptions can include, but are *not limited to* the teacher's expressions, emotions, gestures, speech, and teacher-student, individual, pair-based, or group-based interactions/dynamics if any.

Observable teacher actions include, but are *not limited to* explaining, demonstrating, questioning, answering, responding, prompting, giving hints, correcting, solving, presenting, checking, summarizing, assigning work, monitoring, circulating, assisting, facilitating discussion, or supporting students.

If not enough information exists to make any statements on anything mentioned above, do not produce any statement.

- Detail the students presented in the given frame, concretely describing their actions, when they are visible, audible, or explicitly mentioned. Such descriptions can include, but are *not limited to* the students' expressions, emotions, gestures, speech, and student-teacher, individual, pair-based, or group-based interactions/dynamics if any.

Observable student actions include, but are *not limited to* listening, responding, solving, explaining, presenting, checking, revising, questioning, demonstrating, summarizing, answering, assisting, discussing, or working independently.

If not enough information exists to make any statements on anything mentioned above, do not produce any statement.

- Format your output with level 3 headings, with the headings being Environment, Objects, Teacher, and Student. Under each heading exists the respective description.
'''

In [ ]:
summary_prompt_template = f"""
RESPONSE STYLE
==============
{caveman_prompt}

ROLE
====
You are a classroom observation analyst. You receive a time-ordered window of per-frame descriptions, each broken down by category (ENVIRONMENT, SHENQI, TEACHER, STUDENTS) at 3-second intervals.

Synthesize these fragments into a single coherent narrative of what is happening across the window, captured by the camera. Track continuity and change: who is doing what, how the classroom state evolves, and what the instructional activity is. Weave the categories together rather than listing them separately. Do not use bullet points. Write in flowing paragraphs.

The goal is to assist educators in reviewing what happens in a classroom. Therefore, there is no need to be repetitive with each time-ordered window you receive. That is, if you have already previously described something in an earlier window, do not describe it again in a later window. Unless there is a difference.

Once the teachers'/students'/SHENQIs'/environments' appearances have already been described in an earlier window, do not describe it again unless there is a change.

Do not prefix your response with time range markers such as [0-60s]. The calling system adds these automatically.

Lean towards a narrative, rather than being descriptive.

In this run, this is what you have been requested to observe.

{{}}

Report only what was requested, recalling the role you have been assigned.

Respond in English. Go.

OTHER NOTES
===========
- Produce a concise but information-rich, evidence-grounded, classroom segment description.
- Organize the description chronologically when multiple instructional or interactional phases occur.
- Explicitly describe meaningful transitions when they are observable.
- Use neutral observational language.
- Prefer concrete statements about actors, actions, tasks, and interactions over evaluative language.
- Omit dimensions for which there is no sufficient evidence.
- Focus on the current segment rather than making claims about the entire lesson, unless the evidence supports otherwise.
- No speculation, no assumptions, no decorative language, no inferences, no inventions, and no observations beyond what was specifically requested. If there is ambiguity, state so.
"""

In [ ]:
perspective_prompts = AttrDict(
    whole_class="""Your primary goal is to capture what is happening across the segment, including teacher actions, student actions, classroom interaction, mathematical activity, instructional organization, and meaningful changes over time.

CORE BEHAVIORS
==============

1. Whole-Class Observation
   Analyze the classroom as an interacting system rather than focusing exclusively on the teacher or individual students.

2. Lesson Phase and Activity
   Identify observable instructional activities such as explanation, questioning, guided work, independent work, pair/group work, discussion, practice, review, presentation, or summary.

3. Teacher and Student Actions
   Describe concrete teacher actions and student actions when they are visible, audible, or explicitly stated.

4. Interaction Organization
   Pay attention to whether interaction is whole-class, teacher-student, individual, pair-based, or group-based.

5. Temporal Development
   Track changes in classroom organization and instructional activity over time rather than compressing the entire segment into one general description.

6. Mathematical Activity
   Describe the mathematical topic, task, procedure, representation, or solution strategy when supported by the evidence.

7. Participation vs. Intellectual Initiative
   Distinguish observable participation from intellectual initiative. Do not assume that answering teacher questions means that students independently developed the reasoning.

8. Evidence of Difficulty or Understanding
   Report observable evidence of student understanding, difficulty, uncertainty, errors, questions, or correction when available. Do not infer student understanding from teacher explanation alone.
""",
    teacher="""Your goal is to describe what the teacher does, says, presents, asks, monitors, corrects, supports, or changes during the segment, and how these actions relate to observable student activity.

CORE BEHAVIORS
==============

1. Teacher Instructional Actions
   Identify observable teacher actions such as explaining, demonstrating, questioning, prompting, giving hints, correcting, summarizing, assigning work, monitoring, circulating, or supporting students.

2. Teacher Questioning and Scaffolding
   Describe how the teacher responds to student answers, questions, errors, difficulties, or uncertainty.

3. Degree of Guidance
   Distinguish between direct teacher explanation and situations in which the teacher allows students to explore, propose, test, or revise their own approaches.

4. Response to Student Difficulties
   Report observable cases where the teacher notices or responds to confusion, mistakes, different rates of progress, or requests for assistance.

5. Instructional Organization
   Track teacher-directed transitions between whole-class instruction, independent work, pair/group work, discussion, presentation, or review.

6. Use of Resources
   Describe observable use of the board, worksheets, projector, textbook, diagrams, physical objects, or other instructional resources and how they are used in the activity.

7. Mathematical Guidance
   Describe mathematical explanations, procedures, representations, solution strategies, or questions introduced by the teacher when supported by evidence.

8. Teacher Action vs. Student Outcome
   Do not infer student understanding from teacher explanation alone. Clearly distinguish what the teacher provided from what students demonstrably did.
""",
    researcher="""Your task is to produce a systematic, evidence-grounded description of the instructional structure of the classroom segment, with particular attention to interaction patterns, task organization, mathematical activity, and changes in instructional organization over time.

CORE BEHAVIORS
==============

1. Instructional Structure
   Identify observable lesson phases such as introduction, review, explanation, problem solving, practice, independent work, group work, public discussion, presentation, or summary.

2. Interaction Patterns
   Characterize observable interaction as whole-class/public interaction, teacher-student interaction, individual/private work, pair work, or group work.

3. Transitions
   Track shifts between interaction structures and instructional activities. Do not compress a multi-phase segment into one general classroom description.

4. Task Organization
   Describe how mathematical tasks are introduced, assigned, worked on, discussed, checked, or summarized.

5. Teacher Role
   Describe observable teacher behavior such as directing activity, circulating, questioning, explaining, monitoring, assisting, or facilitating discussion.

6. Student Role
   Describe whether students are listening, responding, solving, discussing, explaining, presenting, checking, revising, or working independently.

7. Student Intellectual Initiative
   Distinguish teacher-directed participation from student-generated mathematical reasoning or solution strategies. Do not assume that responding to teacher prompts constitutes independent reasoning.

8. Mathematical Reasoning and Solution Methods
   Identify observable procedures, representations, solution strategies, alternative approaches, comparisons between methods, or explanations of reasoning.

9. Instructional Resources
   Describe observable use of worksheets, board work, diagrams, projector, textbooks, physical objects, or other representations.

10. Observable Difficulty and Support
    Report observable student errors, confusion, uncertainty, questions, or different levels of progress, together with relevant teacher responses.
""",
    nrc="""Your task is to produce an evidence-grounded description of what mathematical work students are engaged in, what difficulties or decisions arise, how solution strategies develop, and how the teacher supports this process.

CORE BEHAVIORS
==============

1. Mathematical Task
   Identify the mathematical problem, concept, procedure, representation, or application being addressed when it is observable or explicitly stated.

2. Task Demands
   Describe what students are required to determine, calculate, represent, compare, explain, or discover.

3. Student Reasoning
   Report observable student ideas, solution strategies, hypotheses, calculations, representations, explanations, or revisions.

4. Intellectual Initiative
   Distinguish between reasoning generated by students and reasoning supplied through teacher questioning or explanation. Do not assume that answering teacher questions means that students independently developed the reasoning.

5. Difficulties and Errors
   Pay particular attention to observable mathematical difficulties, misconceptions, incorrect approaches, uncertainty, or incomplete reasoning.

6. Teacher Support for Mathematical Thinking
   Describe how the teacher responds to mathematical difficulties through questions, hints, explanations, feedback, representations, or other support.

7. Development of the Solution
   Track how the mathematical activity develops over time, including changes in strategies, corrections, intermediate steps, or movement toward a solution.

8. Alternative Strategies
   Report multiple solution methods or representations when they are actually proposed or demonstrated.

9. Use of Mathematical Resources
   Describe how diagrams, worksheets, formulas, manipulatives, physical objects, board representations, or other resources contribute to observable mathematical work.

10. Evidence of Understanding
    Report observable evidence such as successful explanation, correct application, self-correction, or justified reasoning when available. Do not infer student understanding from teacher explanation alone.
""",
)

In [ ]:
summary_prompts = AttrDict(
    whole_class=summary_prompt_template.format(perspective_prompts.whole_class),
    teacher=summary_prompt_template.format(perspective_prompts.teacher),
    researcher=summary_prompt_template.format(perspective_prompts.researcher),
    nrc=summary_prompt_template.format(perspective_prompts.nrc),
)

In [ ]:
countries = ['JP', 'NL']
selected = {}
for c in countries:
    m_vid = L(db.t.video()).filter(lambda v: v.title.startswith(f'M-{c}'))[0]
    s_vid = L(db.t.video()).filter(lambda v: v.title.startswith(f'S-{c}'))[0]
    selected[f'M-{c}'] = m_vid.id
    selected[f'S-{c}'] = s_vid.id
selected

{'M-JP': 13, 'S-JP': 39, 'M-NL': 17, 'S-NL': 44}

In [ ]:
for label, vid in selected.items():
    print(f'\n{"="*60}\n=== {label} (video_id={vid}) ===\n{"="*60}')
    for model_key in ['seed2p0mini', 'gemma31b']:
        s = session(system=caveman_prompt, model=models[model_key].name, display=False, **models[model_key].kw)
        await deploy_run(vid, db, s, vlm_prompt, prompt_type='default', stop=600, step=5, cache=True, n_workers=20, pause=0.1, max_retries=2)



=== M-JP (video_id=13) ===


!! Using cache


╭─ Run #1 ═══════════════════════════════╮
│ Model    bytedance-seed/seed-2.0-mini
│ Start    0
│ Stop     600
│ Step     5
│ Frames   120
│ Cache    True
│ Time     11:01:36
╰──────────────────────────────────────────────╯


1/120

2/120

...

31/120

32/120

33/120

!! APIError sc=None frame 191: APIError(message='ConnectTimeout', endpoint='POST /api/v1/chat/completions', error_type='ConnectTimeout', code='ConnectTimeout', retryable=True, raw=ConnectTimeout(''))


!! Skipped: frame 191


In [ ]:
db.t.run[1]

Run(id=1, deploy_time='2026-08-27T11:01:36.685447+08:00', finish_time='2026-08-27T13:04:06.263468+08:00', start_time='2026-08-27T11:01:36.685447+08:00', total_duration='2:02:29', video_id=13, model='bytedance-seed/seed-2.0-mini', usage='{"prompt_tokens": 231011, "completion_tokens": 159935, "total_tokens": 390946, "cost": 0.08707510000000002, "is_byok": 0, "prompt_tokens_details": {"cached_tokens": 0, "cache_write_tokens": 0, "audio_tokens": 0, "video_tokens": 0}, "cost_details": {"upstream_inference_cost": 0.08707510000000002, "upstream_inference_prompt_cost": 0.023101099999999986, "upstream_inference_completions_cost": 0.06397400000000004}, "completion_tokens_details": {"reasoning_tokens": 135742, "image_tokens": 0, "audio_tokens": 0}}', num_frames=120, start_sec=0, end_sec=600, step=5, description=None)

In [ ]:
s = session(system=caveman_prompt, model=models['seed2p0mini'].name, display=False, **models['seed2p0mini'].kw)

In [ ]:
for p in summary_prompts.values(): await summarize_run(db, s, sys_prompt=p, run_id=1, window_sec=60, step=1, cache=True)

In [ ]:
for label, vid in selected.items():
    for model_key in ['seed2p0mini', 'kimi3', 'gemma31b']:
        model_name = models[model_key].name
        run_ids = L(db.t.run('video_id=? AND model=?', (vid, model_name))).map(lambda r: r.id).list()
        if not run_ids:
            print(f'No runs for {label} | {model_key}')
            continue
        for perspective_key, sys_prompt in summary_prompts.items():
            s = session(model=model_name, display=False, **models[model_key].kw)
            full_summary = ''
            for start_sec in range(0, 900, 60):
                stop_sec = min(start_sec + 59, 899)
                summary = await summarize_window(db, run_ids, start_sec, stop_sec, s, sys_prompt, cache=True, context=full_summary)
                heading = f'[{start_sec}–{stop_sec}s]'
                summary = summary.strip()
                full_summary = full_summary + f'\n\n{heading} {summary}' if full_summary else f'{heading} {summary}' if summary else ''
            print(f'\n{"="*60}\n=== {label} | {model_key} | {perspective_key} ===\n{"="*60}')
            print(full_summary)


## Time and Cost Calculations

| Model          | Step | 15min $ | 15min ⏱ | 30min $ | 30min ⏱ | 45min $ | 45min ⏱  | 1hr $  | 1hr ⏱    |
| -------------- | ---- | ------- | ------- | ------- | ------- | ------- | -------- | ------ | -------- |
| seed2.0-mini   | s3   | $0.38   | 1h20m   | $0.76   | 2h41m   | $1.14   | 4h02m    | $1.52  | 5h22m    |
| seed2.0-mini   | s2   | $0.59   | 2h05m   | $1.17   | 4h09m   | $1.76   | 6h14m    | $2.34  | 8h18m    |
| kimi-k3        | s2   | $13.69  | 11h34m  | $27.37  | 23h08m  | $41.06  | 1d10h42m | $54.74 | 1d22h16m |
| kimi-k3        | s3   | $8.35   | 7h17m   | $16.70  | 14h33m  | $25.05  | 21h50m   | $33.40 | 1d05h06m |
| gemma-4-31b    | s2   | $0.41   | 9h00m   | $0.82   | 18h00m  | $1.23   | 1d03h00m | $1.64  | 1d12h00m |
| gemma-4-31b    | s3   | $0.24   | 5h26m   | $0.49   | 10h52m  | $0.73   | 16h18m   | $0.97  | 21h44m   |
| qwen3.7-plus   | s3   | $1.33   | 4h49m   | $2.65   | 9h37m   | $3.98   | 14h26m   | $5.30  | 19h14m   |
| glm-4.6v       | s2   | $0.94   | 4h51m   | $1.87   | 9h41m   | $2.81   | 14h32m   | $3.74  | 19h22m   |
| glm-4.6v       | s3   | $0.51   | 2h48m   | $1.03   | 5h36m   | $1.54   | 8h24m    | $2.05  | 11h12m   |
| minimax-m3     | s2   | $0.87   | 3h19m   | $1.74   | 6h38m   | $2.61   | 9h57m    | $3.48  | 13h16m   |
| minimax-m3     | s3   | $0.53   | 2h11m   | $1.05   | 4h21m   | $1.58   | 6h32m    | $2.10  | 8h42m    |
| step-3.7-flash | s2   | $2.06   | 3h54m   | $4.11   | 7h48m   | $6.17   | 11h42m   | $8.22  | 15h36m   |
| step-3.7-flash | s3   | $1.76   | 4h11m   | $3.52   | 8h22m   | $5.27   | 12h33m   | $7.03  | 16h44m   |
| mimo-v2.5      | s3   | $0.26   | 8h18m   | $0.51   | 16h35m  | $0.77   | 1d00h53m | $1.02  | 1d09h10m |
| mistral-small  | s2   | $0.35   | 1h25m   | $0.69   | 2h50m   | $1.04   | 4h15m    | $1.38  | 5h40m    |
| mistral-small  | s3   | $0.21   | 54m     | $0.43   | 1h48m   | $0.64   | 2h42m    | $0.85  | 3h36m    |
| grok-4.5       | s2   | $6.77   | 5h32m   | $13.54  | 11h04m  | $20.30  | 16h36m   | $27.07 | 22h08m   |
| grok-4.5       | s3   | $3.96   | 3h37m   | $7.91   | 7h13m   | $11.87  | 10h50m   | $15.83 | 14h26m   |

Excluded the two errored runs (qwen s2, mimo s2).

Cost winners: **mistral-small s3** ($0.21–$0.85) and **gemma-4-31b s3** ($0.24–$0.97) dominate. Speed winner: **mistral-small s3** (54min–3h36m). kimi-k3 is wildly expensive ($8–$55) and slow. grok-4.5 also pricey ($4–$27). Note step-3.7-flash s3 is *slower* than s2 despite fewer frames — interesting per-frame overhead.

1. **从每次成功的运行输出中提取数据** — 我查看了每个运行的 `Frames`、`Cost` 和 `Elapsed` 数值，忽略了两次失败的情况（qwen s2 和 mimo s2）。
2. **计算每帧的成本 (cpf)** — `cpf = cost / frames`。例如，mistral-small s3：$0.0071 / 10 = $0.00071 per frame。
3. **计算每帧的时间 (tpf)** — `tpf = elapsed_seconds / frames`。例如，mistral-small s3：108s / 10 = 10.8s per frame。
4. **根据步长确定视频的帧数** — 在 `sample_rate=1`（每秒 1 帧）的情况下，15 分钟的视频有 900 帧。在 `step=3` 时，处理 `900 // 3 = 300` 帧。在 `step=2` 时，处理 `900 // 2 = 450` 帧。
5. **乘以估计值** — `cost = num_frames × cpf`，`time = num_frames × tpf`，分别针对 15 分钟（900s）、30 分钟（1800s）、45 分钟（2700s）、1 小时（3600s）的视频进行计算。

一个假设是：各模型之间的 **每帧成本和时间保持不变** — 即扩展是线性的。这是一个一阶近似值；实际运行可能会有所不同，原因包括 API 延迟波动、缓存效应或内容触发的速率限制（这正是导致 qwen 和 mimo 失败的原因）。

- I can also some how implement embeddings/rag system into all of this
- I can implement fastcore.parallel into `deploy_run` itself? Would I get rate limited for doing so? (Search the open router docs for info on this.)


In [ ]:
from vlm_monitor.core import *
from cachy import enable_cachy; enable_cachy()

!rm -rf db_demo.db
db = init_db('db_demo.db')
dpath = Path('../data/timss/')
dpaths = filter_paths(dpath.ls()).sorted(lambda o: (o.stem[:-1], o.stem[-1]))
populate_db(db, dpaths)
len(db.t.video()), len(db.t.frame())

In [ ]:
models = AttrDict(
    seed2p0mini=AttrDict(name='bytedance-seed/seed-2.0-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi3=AttrDict(name='moonshotai/kimi-k3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    gemma31b=AttrDict(name='google/gemma-4-31b-it', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
)

In [ ]:
caveman_prompt = '''
Respond terse like smart caveman. All technical substance stay. Only fluff die.

## Persistence

ACTIVE EVERY RESPONSE. No revert after many turns. No filler drift. Still active if unsure.

## Rules

Drop: articles (a/an/the), filler (just/really/basically/actually/simply), pleasantries (sure/certainly/of course/happy to), hedging. Fragments OK. Short synonyms (big not extensive, fix not "implement a solution for"). Technical terms exact. Code blocks unchanged. Errors quoted exact.

Pattern: `[thing] [action] [reason]. [next step.]`

Not: "Sure! I'd be happy to help you with that. The issue you're experiencing is likely caused by..."
Yes: "Bug in auth middleware. Token expiry check use `<` not `<=`. Fix:"

## Intensity

Example — "Why React component re-render?"
- "New object ref each render. Inline object prop = new ref = re-render. Wrap in `useMemo`."

Example — "Explain database connection pooling."
- "Pool reuse open DB connections. No new connection per request. Skip handshake overhead."

## Auto-Clarity

Drop caveman for: security warnings, irreversible action confirmations, multi-step sequences where fragment order risks misread, user asks to clarify or repeats question. Resume caveman after clear part done.

Example — destructive op:
> **Warning:** This will permanently delete all rows in the `users` table and cannot be undone.
> ```sql
> DROP TABLE users;
> ```
> Caveman resume. Verify backup exist first.
'''

prompt_template = '''
You are an observer of a classroom. You will be provided with a recording of a class, and your task at hand is to neutrally describe the requested feature of the classroom. You need to ground the descriptions with evidence supported by the supplied frame, subtitle, or other input provided to you.

Here, you are assigned to observe:

{}

Report only what is directly observable, and what was requested, recalling the role you have been assigned.

No speculation, no assumptions, no decorative language, no conclusions, no inferences, no inventions, no external knowledge and no observations beyond what was specifically requested.

If there is ambiguity, state so.
'''

prompts = AttrDict(
    environment=prompt_template.format('Detail the environment presented in the given frame.'),
    shenqi=prompt_template.format('Describe observable use of the whiteboard, blackboard, worksheets, projector, textbooks, computers, diagrams, physical objects, or other instructional resources and how they are being used. If these objects contain contents, describe the topic, task, procedure, problem, concept, application, representation, or solution strategy present on the contents.'),
    teacher=prompt_template.format("""
    Detail the teacher presented in the given frame, concretely describing their actions, when they are visible, audible, or explicitly mentioned. Such descriptions can include, but are *not limited to* the teacher's expressions, emotions, gestures, speech, and teacher-student, individual, pair-based, or group-based interactions/dynamics if any.

    Observable teacher actions include, but are *not limited to* explaining, demonstrating, questioning, answering, responding, prompting, giving hints, correcting, solving, presenting, checking, summarizing, assigning work, monitoring, circulating, assisting, facilitating discussion, or supporting students.

    If not enough information exists to make any statements on anything mentioned above, do not produce any statement.
    """),
    students=prompt_template.format("""
    Detail the students presented in the given frame, concretely describing their actions, when they are visible, audible, or explicitly mentioned. Such descriptions can include, but are *not limited to* the students' expressions, emotions, gestures, speech, and student-teacher, individual, pair-based, or group-based interactions/dynamics if any.

    Observable student actions include, but are *not limited to* listening, responding, solving, explaining, presenting, checking, revising, questioning, demonstrating, summarizing, answering, assisting, discussing, or working independently.

    If not enough information exists to make any statements on anything mentioned above, do not produce any statement.
    """),
)


In [ ]:
perspective_prompts = AttrDict(
    whole_class="""Your primary goal is to capture what is happening across the segment, including teacher actions, student actions, classroom interaction, mathematical activity, instructional organization, and meaningful changes over time.

CORE BEHAVIORS
==============

1. Whole-Class Observation
   Analyze the classroom as an interacting system rather than focusing exclusively on the teacher or individual students.

2. Lesson Phase and Activity
   Identify observable instructional activities such as explanation, questioning, guided work, independent work, pair/group work, discussion, practice, review, presentation, or summary.

3. Teacher and Student Actions
   Describe concrete teacher actions and student actions when they are visible, audible, or explicitly stated.

4. Interaction Organization
   Pay attention to whether interaction is whole-class, teacher-student, individual, pair-based, or group-based.

5. Temporal Development
   Track changes in classroom organization and instructional activity over time rather than compressing the entire segment into one general description.

6. Mathematical Activity
   Describe the mathematical topic, task, procedure, representation, or solution strategy when supported by the evidence.

7. Participation vs. Intellectual Initiative
   Distinguish observable participation from intellectual initiative. Do not assume that answering teacher questions means that students independently developed the reasoning.

8. Evidence of Difficulty or Understanding
   Report observable evidence of student understanding, difficulty, uncertainty, errors, questions, or correction when available. Do not infer student understanding from teacher explanation alone.
""",
    teacher="""Your goal is to describe what the teacher does, says, presents, asks, monitors, corrects, supports, or changes during the segment, and how these actions relate to observable student activity.

CORE BEHAVIORS
==============

1. Teacher Instructional Actions
   Identify observable teacher actions such as explaining, demonstrating, questioning, prompting, giving hints, correcting, summarizing, assigning work, monitoring, circulating, or supporting students.

2. Teacher Questioning and Scaffolding
   Describe how the teacher responds to student answers, questions, errors, difficulties, or uncertainty.

3. Degree of Guidance
   Distinguish between direct teacher explanation and situations in which the teacher allows students to explore, propose, test, or revise their own approaches.

4. Response to Student Difficulties
   Report observable cases where the teacher notices or responds to confusion, mistakes, different rates of progress, or requests for assistance.

5. Instructional Organization
   Track teacher-directed transitions between whole-class instruction, independent work, pair/group work, discussion, presentation, or review.

6. Use of Resources
   Describe observable use of the board, worksheets, projector, textbook, diagrams, physical objects, or other instructional resources and how they are used in the activity.

7. Mathematical Guidance
   Describe mathematical explanations, procedures, representations, solution strategies, or questions introduced by the teacher when supported by evidence.

8. Teacher Action vs. Student Outcome
   Do not infer student understanding from teacher explanation alone. Clearly distinguish what the teacher provided from what students demonstrably did.
""",
    researcher="""Your task is to produce a systematic, evidence-grounded description of the instructional structure of the classroom segment, with particular attention to interaction patterns, task organization, mathematical activity, and changes in instructional organization over time.

CORE BEHAVIORS
==============

1. Instructional Structure
   Identify observable lesson phases such as introduction, review, explanation, problem solving, practice, independent work, group work, public discussion, presentation, or summary.

2. Interaction Patterns
   Characterize observable interaction as whole-class/public interaction, teacher-student interaction, individual/private work, pair work, or group work.

3. Transitions
   Track shifts between interaction structures and instructional activities. Do not compress a multi-phase segment into one general classroom description.

4. Task Organization
   Describe how mathematical tasks are introduced, assigned, worked on, discussed, checked, or summarized.

5. Teacher Role
   Describe observable teacher behavior such as directing activity, circulating, questioning, explaining, monitoring, assisting, or facilitating discussion.

6. Student Role
   Describe whether students are listening, responding, solving, discussing, explaining, presenting, checking, revising, or working independently.

7. Student Intellectual Initiative
   Distinguish teacher-directed participation from student-generated mathematical reasoning or solution strategies. Do not assume that responding to teacher prompts constitutes independent reasoning.

8. Mathematical Reasoning and Solution Methods
   Identify observable procedures, representations, solution strategies, alternative approaches, comparisons between methods, or explanations of reasoning.

9. Instructional Resources
   Describe observable use of worksheets, board work, diagrams, projector, textbooks, physical objects, or other representations.

10. Observable Difficulty and Support
    Report observable student errors, confusion, uncertainty, questions, or different levels of progress, together with relevant teacher responses.
""",
    nrc="""Your task is to produce an evidence-grounded description of what mathematical work students are engaged in, what difficulties or decisions arise, how solution strategies develop, and how the teacher supports this process.

CORE BEHAVIORS
==============

1. Mathematical Task
   Identify the mathematical problem, concept, procedure, representation, or application being addressed when it is observable or explicitly stated.

2. Task Demands
   Describe what students are required to determine, calculate, represent, compare, explain, or discover.

3. Student Reasoning
   Report observable student ideas, solution strategies, hypotheses, calculations, representations, explanations, or revisions.

4. Intellectual Initiative
   Distinguish between reasoning generated by students and reasoning supplied through teacher questioning or explanation. Do not assume that answering teacher questions means that students independently developed the reasoning.

5. Difficulties and Errors
   Pay particular attention to observable mathematical difficulties, misconceptions, incorrect approaches, uncertainty, or incomplete reasoning.

6. Teacher Support for Mathematical Thinking
   Describe how the teacher responds to mathematical difficulties through questions, hints, explanations, feedback, representations, or other support.

7. Development of the Solution
   Track how the mathematical activity develops over time, including changes in strategies, corrections, intermediate steps, or movement toward a solution.

8. Alternative Strategies
   Report multiple solution methods or representations when they are actually proposed or demonstrated.

9. Use of Mathematical Resources
   Describe how diagrams, worksheets, formulas, manipulatives, physical objects, board representations, or other resources contribute to observable mathematical work.

10. Evidence of Understanding
    Report observable evidence such as successful explanation, correct application, self-correction, or justified reasoning when available. Do not infer student understanding from teacher explanation alone.
""",
)

summary_prompt_template = f"""
RESPONSE STYLE
==============
{caveman_prompt}

ROLE
====
You are a classroom observation analyst. You receive a time-ordered window of per-frame descriptions, each broken down by category (ENVIRONMENT, SHENQI, TEACHER, STUDENTS) at 3-second intervals.

Synthesize these fragments into a single coherent narrative of what is happening across the window, captured by the camera. Track continuity and change: who is doing what, how the classroom state evolves, and what the instructional activity is. Weave the categories together rather than listing them separately. Do not use bullet points. Write in flowing paragraphs.

The goal is to assist educators in reviewing what happens in a classroom. Therefore, there is no need to be repetitive with each time-ordered window you receive. That is, if you have already previously described something in an earlier window, do not describe it again in a later window. Unless there is a difference.

Once the teachers'/students'/SHENQIs'/environments' appearances have already been described in an earlier window, do not describe it again unless there is a change.

Do not prefix your response with time range markers such as [0-60s]. The calling system adds these automatically.

Lean towards a narrative, rather than being descriptive.

In this run, this is what you have been requested to observe.

{{}}

Report only what was requested, recalling the role you have been assigned.

Respond in English. Go.

OTHER NOTES
===========
- Produce a concise but information-rich, evidence-grounded, classroom segment description.
- Organize the description chronologically when multiple instructional or interactional phases occur.
- Explicitly describe meaningful transitions when they are observable.
- Use neutral observational language.
- Prefer concrete statements about actors, actions, tasks, and interactions over evaluative language.
- Omit dimensions for which there is no sufficient evidence.
- Focus on the current segment rather than making claims about the entire lesson, unless the evidence supports otherwise.
- No speculation, no assumptions, no decorative language, no inferences, no inventions, and no observations beyond what was specifically requested. If there is ambiguity, state so.
"""

summary_prompts = AttrDict(
    whole_class=summary_prompt_template.format(perspective_prompts.whole_class),
    teacher=summary_prompt_template.format(perspective_prompts.teacher),
    researcher=summary_prompt_template.format(perspective_prompts.researcher),
    nrc=summary_prompt_template.format(perspective_prompts.nrc),
)


In [ ]:
countries = ['HK', 'JP', 'AU', 'NL']
selected = {}
for c in countries:
    m_vid = L(db.t.video()).filter(lambda v: v.title.startswith(f'M-{c}'))[0]
    s_vid = L(db.t.video()).filter(lambda v: v.title.startswith(f'S-{c}'))[0]
    selected[f'M-{c}'] = m_vid.id
    selected[f'S-{c}'] = s_vid.id
selected


In [ ]:
for label, vid in selected.items():
    print(f'\n{"="*60}\n=== {label} (video_id={vid}) ===\n{"="*60}')
    for model_key in ['seed2p0mini', 'kimi3', 'gemma31b']:
        s = session(system=caveman_prompt, model=models[model_key].name, display=False, **models[model_key].kw)
        for ptype, prompt in prompts.items():
            print(f'\n--- {label} | {model_key} | {ptype} ---')
            await deploy_run(vid, db, s, prompt, prompt_type=ptype, stop=900, cache=False, n_workers=20, pause=0.1, max_retries=2)


In [ ]:
for label, vid in selected.items():
    for model_key in ['seed2p0mini', 'kimi3', 'gemma31b']:
        model_name = models[model_key].name
        run_ids = L(db.t.run('video_id=? AND model=?', (vid, model_name))).map(lambda r: r.id).list()
        if not run_ids:
            print(f'No runs for {label} | {model_key}')
            continue
        for perspective_key, sys_prompt in summary_prompts.items():
            s = session(model=model_name, display=False, **models[model_key].kw)
            full_summary = ''
            for start_sec in range(0, 900, 60):
                stop_sec = min(start_sec + 59, 899)
                summary = await summarize_window(db, run_ids, start_sec, stop_sec, s, sys_prompt, cache=True, context=full_summary)
                heading = f'[{start_sec}–{stop_sec}s]'
                summary = summary.strip()
                full_summary = full_summary + f'\n\n{heading} {summary}' if full_summary else f'{heading} {summary}' if summary else ''
            print(f'\n{"="*60}\n=== {label} | {model_key} | {perspective_key} ===\n{"="*60}')
            print(full_summary)


In [ ]:
models = AttrDict(
    gemma26b       =AttrDict(name='google/gemma-4-26b-a4b-it:free', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    gemma31b       =AttrDict(name='google/gemma-4-31b-it', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    grok4p3        =AttrDict(name='x-ai/grok-4.3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    grok4p5        =AttrDict(name='x-ai/grok-4.5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    qwen3p7plus    =AttrDict(name='qwen/qwen3.7-plus', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    qwen3p7max     =AttrDict(name='qwen/qwen3.7-max', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mistral3p5     =AttrDict(name='mistralai/mistral-medium-3-5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mistral4       =AttrDict(name='mistralai/mistral-small-2603', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    n2mini         =AttrDict(name='nex-agi/nex-n2-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    n2pro          =AttrDict(name='nex-agi/nex-n2-pro', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    hy3            =AttrDict(name='tencent/hy3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    fugu           =AttrDict(name='sakana/fugu-ultra', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    glm4p6v        =AttrDict(name='z-ai/glm-4.6v', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    glm5p2         =AttrDict(name='z-ai/glm-5.2', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi2p6        =AttrDict(name='moonshotai/kimi-k2.6', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi2p7code    =AttrDict(name='moonshotai/kimi-k2.7-code', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_nano  =AttrDict(name='nvidia/nemotron-3-nano-30b-a3b', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_omni  =AttrDict(name='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_ultra =AttrDict(name='nvidia/nemotron-3-ultra-550b-a55b', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    minimax_m3     =AttrDict(name='minimax/minimax-m3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    step3p7        =AttrDict(name='stepfun/step-3.7-flash', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mimo_v2p5      =AttrDict(name='xiaomi/mimo-v2.5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mimo_v2p5pro   =AttrDict(name='xiaomi/mimo-v2.5-pro', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    seed2p0mini    =AttrDict(name='bytedance-seed/seed-2.0-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    seed2p0lite    =AttrDict(name='bytedance-seed/seed-2.0-lite', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
)